In [ ]:
import mesa
import pandas as pd
import geopandas as gpd
from mesa import model
from mesa.batchrunner import batch_run

from model.traffic_model import TrafficModel
from model.reporting import model_reporters


# Import Road

In [ ]:
# import road
road_gdf = gpd.read_parquet("data/roads/hw210_sl_and_curvs.parquet")
# Expected counts
ecs_df = pd.read_csv('data/vehicle_counts/expected_counts_seconds.csv')

display(road_gdf.head(3))
#au.plot_colored_road(road_gdf, 'curvature')

# Set Up Perams

In [ ]:
# use this to see what the current state of TrafficModel is
# IMPORTANT - batch_run needs to pass all the model perams eg. p_generate=0.2, max_persons=50 ect 


# help(TrafficModel)

tm = TrafficModel(
    # meta perams
    road_gdf=road_gdf, ecs_df=ecs_df, max_steps=10000, batchrun=False, collect_every_n=3,
    # car centric perams
    start_hr=7, traffic_percentile=80, max_persons=5000,
    # canyon closure
    canyon_closures={'closure_step': [1], 'duration': [1], 'road_section': [119]}, 
    #bus centric perams
    bus_interval=0, car_preference=.7,
    crashes_per_100k_vmt_input=8
)

In [ ]:
# Define parameters to sweep - if the peram is passed out of a list it does not come back in the results df
perams ={
        "car_preference": [0.9, 0.5],
        "max_persons":100,
        "batchrun": True,
        "start_hr":7
        "traffic_percentile": [60,70, 80],
        "bus_interval":[30, 60],
        'bus_capacity': 30
        'crashes_per_100k_vmt_input': [5,10]
        }

In [ ]:
# deal with the road_gdf peram
def traffic_model_factory(road_gdf, ecs_df=ecs_df):
    def make_model(**kwargs):
        return TrafficModel(road_gdf=road_gdf, ecs_df=ecs_df, **kwargs)
    return make_model


model_with_road_gdf = traffic_model_factory(road_gdf, ecs_df) # produces a mesa.model with my road_gdf as a existing peram

# Actual batch_run

In [ ]:


if __name__ == '__main__':
    results = batch_run(
        model_with_road_gdf,
        parameters=perams,
        #number_processes=None,
        iterations=1,
        max_steps=40000,
        display_progress=True,
    )



# Batch data analysis

In [ ]:
results_df = pd.DataFrame(results)
results_df

In [ ]:


# some rando analysis from the end of the semester, havent really looked at it recently 
results_df = pd.DataFrame(results)

results_df['bus_utilization'] = results_df.bus_riders/(results_df.bus_counter*results_df.bus_capacity)
results_df['util_flag'] = (results_df['bus_utilization'] > 0.4).astype(int)

results_df['car_counter'] = results_df.person_counter - results_df.bus_riders

results_df.drop(columns=['iteration', 'RunId', 'max_steps','bus_capacity', 'batchrun'], inplace=True)

results_df.to_csv("flagged_batch_results.csv", index=False)

sns.histplot(results_df.bus_utilization)